In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

In [6]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [9]:
# Feature map
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [13]:
tahminler = []

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        # Train: 2002-2021, Validation: 2022, Test: 2023
        train_data = df_target[df_target['yıl'] <= 2021]
        val_data = df_target[df_target['yıl'] == 2022]
        test_data = df_target[df_target['yıl'] == 2023]

        if val_data.empty or test_data.empty:
            continue

        X_train = train_data[features]
        y_train = train_data[hedef]
        X_val = val_data[features]
        y_val = val_data[hedef]
        X_test = test_data[features]
        y_test = test_data[hedef]

        # Model eğitimleri
        lr = LinearRegression().fit(X_train, y_train)
        rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
        xgb = XGBRegressor(objective='reg:squarederror', random_state=42).fit(X_train, y_train)
        svr = SVR().fit(X_train, y_train)
        knn = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)

        # Tahmin ve skorlar
        y_val_pred_lr = lr.predict(X_val)[0]
        y_val_pred_rf = rf.predict(X_val)[0]
        y_val_pred_xgb = xgb.predict(X_val)[0]
        y_val_pred_svr = svr.predict(X_val)[0]
        y_val_pred_knn = knn.predict(X_val)[0]

        y_test_pred_lr = lr.predict(X_test)[0]
        y_test_pred_rf = rf.predict(X_test)[0]
        y_test_pred_xgb = xgb.predict(X_test)[0]
        y_test_pred_svr = svr.predict(X_test)[0]
        y_test_pred_knn = knn.predict(X_test)[0]

        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2022": y_val.values[0],
            "tahmin_2022_lr": y_val_pred_lr,
            "tahmin_2022_rf": y_val_pred_rf,
            "tahmin_2022_xgb": y_val_pred_xgb,
            "tahmin_2022_svr": y_val_pred_svr,
            "tahmin_2022_knn": y_val_pred_knn,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_lr": y_test_pred_lr,
            "tahmin_2023_rf": y_test_pred_rf,
            "tahmin_2023_xgb": y_test_pred_xgb,
            "tahmin_2023_svr": y_test_pred_svr,
            "tahmin_2023_knn": y_test_pred_knn,
        })

In [15]:
df_tahmin = pd.DataFrame(tahminler)

In [17]:
# Doğruluk hesaplama
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    df_tahmin[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df_tahmin["gercek_2023"] - df_tahmin[f"tahmin_2023_{model}"]) / df_tahmin["gercek_2023"] * 100
    )
    df_tahmin[f"{model}_yuzde_dog"] = df_tahmin[f"{model}_yuzde_dog"].clip(0, 100)

# Ortalama doğrulukları yazdır
print("\nŞehir Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ort = round(df_tahmin[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ort}%")

# Doğruluk dosyasını kaydet
df_tahmin.to_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv", index=False)


Şehir Doğruluk Ortalamaları (2023):
 - LR: 67.81%
 - RF: 78.76%
 - XGB: 79.81%
 - SVR: 69.81%
 - KNN: 78.4%


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# MAE, RMSE, MAPE hesaplama
print("\nModel Performans Metrikleri (2023):")
print("{:<6} | {:>10} | {:>10} | {:>10}".format("Model", "MAE", "RMSE", "MAPE (%)"))
print("-" * 44)

metric_sonuclar = []

for model in ["lr", "rf", "xgb", "svr", "knn"]:
    y_true = df_tahmin["gercek_2023"]
    y_pred = df_tahmin[f"tahmin_2023_{model}"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print("{:<6} | {:>10.2f} | {:>10.2f} | {:>10.2f}".format(model.upper(), mae, rmse, mape))

    metric_sonuclar.append({
        "Model": model.upper(),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MAPE (%)": round(mape, 2)
    })

# Metrikleri CSV olarak kaydet
df_metrics = pd.DataFrame(metric_sonuclar)
df_metrics.to_csv("sehir_bazli_2023_model_metrikleri.csv", index=False)



Model Performans Metrikleri (2023):
Model  |        MAE |       RMSE |   MAPE (%)
--------------------------------------------
LR     |     383.91 |    1673.56 |        inf
RF     |     299.14 |    1081.58 |        inf
XGB    |     247.74 |     879.10 |        inf
SVR    |     524.76 |    1936.37 |        inf
KNN    |     317.82 |    1107.94 |        inf


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared err

In [ ]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# Bölge ekle
df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgelere göre grupla ve toplamları al
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[[
    "gercek_2023",
    "tahmin_2023_lr",
    "tahmin_2023_rf",
    "tahmin_2023_xgb",
    "tahmin_2023_svr",
    "tahmin_2023_knn"
]].sum().reset_index()

# CSV olarak kaydet
bolgesel.to_csv("bolge_bazli_2023_tahmin5model.csv", index=False)

In [16]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv")  # Gerekirse yolunu düzenle

# 2. Doğruluk hesaplama fonksiyonu
def hesapla_dog(model_etiketi):
    df[f"{model_etiketi}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model_etiketi}"]) / df["gercek_2023"] * 100
    )
    df[f"{model_etiketi}_yuzde_dog"] = df[f"{model_etiketi}_yuzde_dog"].clip(0, 100)

# 3. Altı model için hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    hesapla_dog(model)

# 4. Ortalamaları yazdır
print("\nŞehir Bazlı Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 5. Güncel dosya olarak kaydet
df.to_csv("sehir_tahmin_yuzdeler.csv", index=False)


Şehir Bazlı Doğruluk Ortalamaları (2023):
 - LR: 67.81%
 - RF: 78.76%
 - XGB: 79.81%
 - SVR: 69.81%
 - KNN: 78.4%


In [21]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin5model.csv")  # Gerekirse dosya adını değiştir

# 2. Her model için yüzdelik doğruluk hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    df[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model}"]) / df["gercek_2023"] * 100
    )
    df[f"{model}_yuzde_dog"] = df[f"{model}_yuzde_dog"].clip(0, 100)

# 3. Ortalamaları hesapla ve yazdır
print(f"\nBölge Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 4. Dosyaya kaydet
df.to_csv("bolge_tahmin_yuzdeler.csv", index=False)


Bölge Doğruluk Ortalamaları (2023):
 - LR: 83.93%
 - RF: 84.77%
 - XGB: 87.4%
 - SVR: 73.15%
 - KNN: 83.72%
